# Proyecto final · Predicción de partidos ATP
## Paso 2 · Datos: de dónde salen y cómo armamos la tabla principal

**Publica:** Jeff Sackmann / Tennis Abstract (tennisabstract.com).
**Licencia:** CC BY-NC-SA 4.0 (atribución, uso no comercial, compartir igual).
**Descarga:** el repositorio original `JeffSackmann/tennis_atp` fue retirado de GitHub en 2026, y la página de Tennis Abstract prohíbe descargar sus archivos de partidos (`robots.txt`). Usamos el espejo **`Aneeshers/tennis-sackmann-archive`** en Hugging Face: mismos archivos y misma licencia, con datos ATP hasta junio de 2026.

### Las cuatro fuentes que unimos

| Tabla | Archivos | Para qué la usamos |
|---|---|---|
| Partidos del tour | `atp_matches_YYYY.csv` | Los partidos que queremos predecir |
| Qualys y challengers | `atp_matches_qual_chall_YYYY.csv` | Historial extra de cada jugador (forma, Elo) |
| Jugadores | `atp_players.csv` | Mano, fecha de nacimiento, país, estatura |
| Rankings semanales | `atp_rankings_00s/10s/20s/current.csv` | Ranking de cada jugador en cada semana |

**Meta del paso:** una tabla principal con **una fila por partido del tour**, armada por nosotros a partir de las cuatro fuentes.

In [2]:
import os
import numpy as np
import pandas as pd

SEMILLA = 42                      # la misma del paso 1, para que el sorteo A/B coincida
ANIO_INICIO, ANIO_FIN = 2000, 2026
ANIO_HISTORIAL = 1995             # el historial arranca antes, para que en 2000 ya haya forma y Elo
CARPETA = os.environ.get("TENIS_DATOS", "datos")
os.makedirs(CARPETA, exist_ok=True)

ESPEJOS = [
    "https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/atp/{}",
    "https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/main/atp/{}",
]

def leer(nombre, obligatorio=True):
    """Lee un CSV ya descargado en la carpeta local."""
    local = os.path.join(CARPETA, nombre)
    if os.path.exists(local):
        return pd.read_csv(local, low_memory=False)
    if obligatorio:
        raise FileNotFoundError(f"Falta {nombre}: vuelva a correr la celda de descarga")
    return None

### Descarga (solo la primera vez)
Baja los archivos en paralelo, con límite de tiempo y reintentos, y muestra el progreso. Los que ya están en `datos/` se saltan, así que desde la segunda vez esta celda tarda segundos.

In [3]:
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

ARCHIVOS = (
    [f"atp_matches_{a}.csv" for a in range(ANIO_HISTORIAL, ANIO_FIN + 1)]
    + [f"atp_matches_qual_chall_{a}.csv" for a in range(ANIO_HISTORIAL, ANIO_FIN + 1)]
    + ["atp_players.csv"]
    + [f"atp_rankings_{d}.csv" for d in ["90s", "00s", "10s", "20s", "current"]]
)

def descargar(nombre, intentos=3):
    local = os.path.join(CARPETA, nombre)
    if os.path.exists(local) and os.path.getsize(local) > 1000:
        return nombre, "ya estaba", 0
    for intento in range(intentos):
        for plantilla in ESPEJOS:
            try:
                r = requests.get(plantilla.format(nombre), timeout=60)
                # GitHub puede devolver un puntero LFS (texto corto) en vez del archivo
                if r.status_code == 200 and len(r.content) > 1000 and not r.content.startswith(b"version https://git-lfs"):
                    with open(local, "wb") as f:
                        f.write(r.content)
                    return nombre, "descargado", len(r.content) / 1e6
                if r.status_code == 404:
                    break
            except requests.RequestException:
                pass
        time.sleep(2 * (intento + 1))
    return nombre, "NO DISPONIBLE", 0

inicio = time.time()
faltan = []
with ThreadPoolExecutor(max_workers=8) as pool:
    tareas = [pool.submit(descargar, a) for a in ARCHIVOS]
    for i, t in enumerate(as_completed(tareas), 1):
        nombre, estado, mb = t.result()
        if estado == "NO DISPONIBLE":
            faltan.append(nombre)
        print(f"[{i:>3}/{len(ARCHIVOS)}] {estado:<13} {nombre} {f'({mb:.1f} MB)' if mb else ''}")
print(f"Listo en {time.time() - inicio:.0f} s. No disponibles: {faltan or 'ninguno'}")

[  1/70] ya estaba     atp_matches_2000.csv 
[  2/70] ya estaba     atp_matches_2001.csv 
[  3/70] ya estaba     atp_matches_2004.csv 
[  4/70] ya estaba     atp_matches_2002.csv 
[  5/70] ya estaba     atp_matches_2006.csv 
[  6/70] ya estaba     atp_matches_2003.csv 
[  7/70] ya estaba     atp_matches_2007.csv 
[  8/70] ya estaba     atp_matches_2008.csv 
[  9/70] ya estaba     atp_matches_2005.csv 
[ 10/70] ya estaba     atp_matches_2011.csv 
[ 11/70] ya estaba     atp_matches_2012.csv 
[ 12/70] ya estaba     atp_matches_2009.csv 
[ 13/70] ya estaba     atp_matches_2014.csv 
[ 14/70] ya estaba     atp_matches_2010.csv 
[ 15/70] ya estaba     atp_matches_2015.csv 
[ 16/70] ya estaba     atp_matches_2013.csv 
[ 17/70] ya estaba     atp_matches_2018.csv 
[ 18/70] ya estaba     atp_matches_2016.csv 
[ 19/70] ya estaba     atp_matches_2019.csv 
[ 20/70] ya estaba     atp_matches_2017.csv 
[ 21/70] ya estaba     atp_matches_2022.csv 
[ 22/70] ya estaba     atp_matches_2020.csv 
[ 23/70] y

### Fuente 1 · Partidos del tour

In [4]:
tour = pd.concat([t for a in range(ANIO_INICIO, ANIO_FIN + 1)
                  if (t := leer(f"atp_matches_{a}.csv", obligatorio=False)) is not None],
                 ignore_index=True)
tour["tourney_date"] = pd.to_datetime(tour["tourney_date"].astype(str), format="%Y%m%d")
print(f"Partidos del tour: {len(tour):,} filas × {tour.shape[1]} columnas")
print(f"Periodo: {tour['tourney_date'].min().date()} a {tour['tourney_date'].max().date()}")
tour.columns.tolist()

Partidos del tour: 79,299 filas × 49 columnas
Periodo: 2000-01-03 a 2026-05-25


['tourney_id',
 'tourney_name',
 'surface',
 'draw_size',
 'tourney_level',
 'tourney_date',
 'match_num',
 'winner_id',
 'winner_seed',
 'winner_entry',
 'winner_name',
 'winner_hand',
 'winner_ht',
 'winner_ioc',
 'winner_age',
 'loser_id',
 'loser_seed',
 'loser_entry',
 'loser_name',
 'loser_hand',
 'loser_ht',
 'loser_ioc',
 'loser_age',
 'score',
 'best_of',
 'round',
 'minutes',
 'w_ace',
 'w_df',
 'w_svpt',
 'w_1stIn',
 'w_1stWon',
 'w_2ndWon',
 'w_SvGms',
 'w_bpSaved',
 'w_bpFaced',
 'l_ace',
 'l_df',
 'l_svpt',
 'l_1stIn',
 'l_1stWon',
 'l_2ndWon',
 'l_SvGms',
 'l_bpSaved',
 'l_bpFaced',
 'winner_rank',
 'winner_rank_points',
 'loser_rank',
 'loser_rank_points']

Una fila es un partido. Las columnas se agrupan en cinco bloques:
- **Torneo:** `tourney_id`, `tourney_name`, `surface`, `draw_size`, `tourney_level`, `tourney_date`, `match_num`, `round`, `best_of`.
- **Ganador / perdedor:** id, nombre, mano, estatura, país, edad, ranking, puntos, cabeza de serie y tipo de entrada.
- **Resultado:** `score` y `minutes`.
- **Estadísticas del partido:** `w_ace`, `w_df`, `w_svpt`, … y lo mismo con `l_`.

Los dos últimos bloques **solo se conocen cuando el partido terminó**. No pueden entrar al modelo tal cual: solo sirven para construir promedios de partidos anteriores.

In [5]:
# ¿El identificador de partido es único?
tour["match_id"] = tour["tourney_id"].astype(str) + "-" + tour["match_num"].astype(str)
duplicados = tour["match_id"].duplicated(keep=False)
print(f"match_id repetidos: {duplicados.sum()} filas")
tour.loc[duplicados, ["match_id", "tourney_name", "round", "winner_name", "loser_name"]].head(10)

match_id repetidos: 0 filas


,match_id,tourney_name,round,winner_name,loser_name


### Fuente 2 · Qualys y challengers (historial adicional)

In [6]:
chall = pd.concat([t for a in range(ANIO_HISTORIAL, ANIO_FIN + 1)
                   if (t := leer(f"atp_matches_qual_chall_{a}.csv", obligatorio=False)) is not None],
                  ignore_index=True)
chall["tourney_date"] = pd.to_datetime(chall["tourney_date"].astype(str), format="%Y%m%d")
print(f"Qualys y challengers: {len(chall):,} filas × {chall.shape[1]} columnas")
print("Niveles:", chall["tourney_level"].value_counts().to_dict())

# Partidos del tour de 1995 a 1999: solo historial, no se predicen
tour_previo = pd.concat([t for a in range(ANIO_HISTORIAL, ANIO_INICIO)
                         if (t := leer(f"atp_matches_{a}.csv", obligatorio=False)) is not None],
                        ignore_index=True)
tour_previo["tourney_date"] = pd.to_datetime(tour_previo["tourney_date"].astype(str), format="%Y%m%d")
print(f"Tour 1995-1999 (solo historial): {len(tour_previo):,} filas")

Qualys y challengers: 207,418 filas × 49 columnas
Niveles: {'C': 175180, 'A': 16505, 'G': 11536, 'M': 4197}
Tour 1995-1999 (solo historial): 18,122 filas


¿Por qué sumar challengers? Muchos jugadores llegan al tour con pocos partidos en él, pero con cientos en challengers. Si solo miramos el tour, su forma y su Elo arrancan "en blanco". Con esta tabla, el historial de cada jugador queda más completo.

### Fuente 3 · Jugadores

In [7]:
jugadores = leer("atp_players.csv")
jugadores["dob"] = pd.to_datetime(jugadores["dob"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
print(f"Jugadores: {len(jugadores):,} filas × {jugadores.shape[1]} columnas")
print("¿player_id único?", jugadores["player_id"].is_unique)
print("Faltantes (%):")
print((jugadores.isna().mean() * 100).round(1).to_string())
jugadores.head()

Jugadores: 66,912 filas × 8 columnas
¿player_id único? True
Faltantes (%):
player_id       0.0
name_first      1.4
name_last       0.1
hand            0.0
dob            28.0
ioc             1.0
height         93.8
wikidata_id    92.7


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,100001,Gardnar,Mulloy,R,1913-11-22,USA,185.0,Q54544
1,100002,Pancho,Segura,R,1921-06-20,ECU,168.0,Q54581
2,100003,Frank,Sedgman,R,1927-10-02,AUS,180.0,Q962049
3,100004,Giuseppe,Merlo,R,1927-10-11,ITA,NaN,Q1258752
4,100005,Richard,Gonzalez,R,1928-05-09,USA,188.0,Q53554


### Fuente 4 · Rankings semanales

In [8]:
rankings = pd.concat([leer(f"atp_rankings_{d}.csv") for d in ["90s", "00s", "10s", "20s", "current"]],
                     ignore_index=True)
rankings["ranking_date"] = pd.to_datetime(rankings["ranking_date"].astype(str), format="%Y%m%d")
antes = len(rankings)
rankings = rankings.drop_duplicates(["ranking_date", "player"])
print(f"Rankings: {len(rankings):,} filas × {rankings.shape[1]} columnas "
      f"({antes - len(rankings):,} duplicadas entre archivos eliminadas)")
print(f"Semanas: {rankings['ranking_date'].nunique():,}, "
      f"de {rankings['ranking_date'].min().date()} a {rankings['ranking_date'].max().date()}")
rankings.head()

Rankings: 3,114,367 filas × 4 columnas (693 duplicadas entre archivos eliminadas)
Semanas: 1,776, de 1990-01-01 a 2026-06-08


,ranking_date,rank,player,points
0,1990-01-01,1,100656,2913.0
1,1990-01-01,2,101414,2279.0
2,1990-01-01,3,101222,2111.0
3,1990-01-01,4,100763,1398.0
4,1990-01-01,5,100581,1354.0


### Armado de la tabla principal

1. **Sorteo A/B** (igual que en el paso 1, misma semilla): la etiqueta es `gana_A`.
2. **Unión con jugadores** por `player_id`: mano, fecha de nacimiento, país y estatura de A y de B.
3. **Unión con rankings "hacia atrás"**: para cada jugador se toma el último ranking publicado **en o antes** de la fecha del torneo. Nunca uno posterior, porque sería una fuga.

`merge_asof(direction="backward")` hace justo eso: busca la fecha más cercana hacia atrás.

In [9]:
rng = np.random.default_rng(SEMILLA)
ganador_es_A = rng.random(len(tour)) < 0.5

principal = pd.DataFrame({
    "match_id": tour["match_id"],
    "tourney_id": tour["tourney_id"],
    "torneo": tour["tourney_name"],
    "fecha": tour["tourney_date"],
    "anio": tour["tourney_date"].dt.year,
    "superficie": tour["surface"],
    "nivel": tour["tourney_level"],
    "cuadro": tour["draw_size"],
    "ronda": tour["round"],
    "best_of": tour["best_of"],
    "score": tour["score"],
    "minutos": tour["minutes"],
})

def lado(col_w, col_l):
    return (np.where(ganador_es_A, tour[col_w], tour[col_l]),
            np.where(ganador_es_A, tour[col_l], tour[col_w]))

# Datos del jugador que trae el archivo de partidos
for col in ["id", "name", "hand", "ht", "ioc", "age", "rank", "rank_points", "seed", "entry"]:
    principal[f"A_{col}"], principal[f"B_{col}"] = lado(f"winner_{col}", f"loser_{col}")

# Estadísticas del partido: SOLO para construir historial en el paso 4, nunca como entrada directa
ESTADISTICAS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]
for col in ESTADISTICAS:
    principal[f"A_{col}"], principal[f"B_{col}"] = lado(f"w_{col}", f"l_{col}")

principal["gana_A"] = ganador_es_A.astype(int)
principal["A_id"] = principal["A_id"].astype("int64")
principal["B_id"] = principal["B_id"].astype("int64")
print(f"Tabla principal: {principal.shape[0]:,} filas × {principal.shape[1]} columnas")

Tabla principal: 79,299 filas × 51 columnas


#### Unión con la tabla de jugadores

In [10]:
info = jugadores.rename(columns={"player_id": "id", "hand": "mano_j", "dob": "nacimiento",
                                 "ioc": "pais_j", "height": "estatura_j"})[
    ["id", "mano_j", "nacimiento", "pais_j", "estatura_j"]]

for L in ["A", "B"]:
    principal = principal.merge(info.add_prefix(f"{L}_"), on=f"{L}_id", how="left", validate="many_to_one")

# ¿La tabla de jugadores completa la estatura que falta en el archivo de partidos?
for L in ["A", "B"]:
    falta = principal[f"{L}_ht"].isna()
    se_llena = falta & principal[f"{L}_estatura_j"].notna()
    print(f"{L}: estatura faltante en partidos {falta.mean():.1%} → la tabla de jugadores llena {se_llena.sum():,} filas")

# Edad recalculada desde la fecha de nacimiento, para validar la columna de edad
dif_edad = ((principal["fecha"] - principal["A_nacimiento"]).dt.days / 365.25 - principal["A_age"]).abs()
print(f"Edad: diferencia mediana entre la calculada y la del archivo = {dif_edad.median():.3f} años")

A: estatura faltante en partidos 2.9% → la tabla de jugadores llena 5 filas
B: estatura faltante en partidos 2.9% → la tabla de jugadores llena 2 filas
Edad: diferencia mediana entre la calculada y la del archivo = 0.052 años


#### Unión con los rankings semanales (hacia atrás en el tiempo)

In [11]:
rk = (rankings.rename(columns={"player": "id", "rank": "rk_unido", "points": "pts_unido"})
        .sort_values("ranking_date"))

def unir_ranking(tabla, L):
    izq = (tabla[["match_id", "fecha", f"{L}_id"]].rename(columns={f"{L}_id": "id"})
           .reset_index().sort_values("fecha"))
    unido = pd.merge_asof(izq, rk, left_on="fecha", right_on="ranking_date",
                          by="id", direction="backward")
    unido = unido.set_index("index").sort_index()
    tabla[f"{L}_rank_unido"] = unido["rk_unido"]
    tabla[f"{L}_pts_unido"] = unido["pts_unido"]
    tabla[f"{L}_dias_ranking"] = (unido["fecha"] - unido["ranking_date"]).dt.days
    return tabla

for L in ["A", "B"]:
    principal = unir_ranking(principal, L)

ambos = principal["A_rank"].notna() & principal["A_rank_unido"].notna()
coincide = (principal.loc[ambos, "A_rank"] == principal.loc[ambos, "A_rank_unido"]).mean()
print(f"Ranking del archivo de partidos = ranking unido en {coincide:.1%} de los partidos")
llena = principal["A_rank"].isna() & principal["A_rank_unido"].notna()
print(f"Partidos donde A no tenía ranking y la unión lo encontró: {llena.sum():,}")
print("Antigüedad del ranking usado (días):")
print(principal["A_dias_ranking"].describe().round(1).to_string())

Ranking del archivo de partidos = ranking unido en 95.1% de los partidos
Partidos donde A no tenía ranking y la unión lo encontró: 458
Antigüedad del ranking usado (días):
count    78711.0
mean         5.1
std         86.9
min          0.0
25%          0.0
50%          0.0
75%          0.0
max       7935.0


Si el ranking unido coincide casi siempre con el del archivo, la unión está bien hecha y confirma que el ranking del archivo es pre-partido. Los casos en que el ranking unido tiene muchos días de antigüedad son jugadores que salieron del ranking y volvieron. En la limpieza decidimos si confiamos en un ranking tan viejo.

### Historial de partidos (para el paso 4)

In [12]:
cols = ["tourney_id", "tourney_name", "surface", "tourney_level", "tourney_date", "match_num", "round",
        "best_of", "winner_id", "loser_id", "score", "minutes"] + \
       [f"w_{c}" for c in ESTADISTICAS] + [f"l_{c}" for c in ESTADISTICAS]
historial = pd.concat([tour_previo[cols].assign(origen="tour"),
                       tour[cols].assign(origen="tour"),
                       chall[cols].assign(origen="qual_chall")], ignore_index=True)
historial = historial.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)
print(f"Historial: {len(historial):,} partidos "
      f"({(historial['origen'] == 'tour').sum():,} del tour + {(historial['origen'] == 'qual_chall').sum():,} de qualys/challengers)")
print(f"Jugadores distintos en el historial: {pd.unique(historial[['winner_id', 'loser_id']].values.ravel()).size:,}")

Historial: 304,839 partidos (97,421 del tour + 207,418 de qualys/challengers)
Jugadores distintos en el historial: 9,048


### Guardar las tablas armadas

In [13]:
principal.to_csv(os.path.join(CARPETA, "tabla_principal.csv.gz"), index=False)
historial.to_csv(os.path.join(CARPETA, "historial_partidos.csv.gz"), index=False)
print("Guardadas en", CARPETA)

Guardadas en datos


### Resumen para la exposición

In [14]:
resumen = pd.DataFrame([
    ["Partidos del tour", "Un partido", len(tour), tour.shape[1], "tourney_id + match_num"],
    ["Qualys y challengers", "Un partido", len(chall), chall.shape[1], "tourney_id + match_num"],
    ["Jugadores", "Un jugador", len(jugadores), jugadores.shape[1], "player_id"],
    ["Rankings", "Un jugador en una semana", len(rankings), rankings.shape[1], "ranking_date + player"],
    ["Tabla principal (nuestra)", "Un partido del tour", len(principal), principal.shape[1], "match_id"],
], columns=["Tabla", "Una fila es", "Filas", "Columnas", "Llave"])
resumen["Filas"] = resumen["Filas"].map("{:,}".format)
resumen

,Tabla,Una fila es,Filas,Columnas,Llave
0,Partidos del tour,Un partido,"79,299",50,tourney_id + match_num
1,Qualys y challengers,Un partido,"207,418",49,tourney_id + match_num
2,Jugadores,Un jugador,"66,912",8,player_id
3,Rankings,Un jugador en una semana,"3,114,367",4,ranking_date + player
4,Tabla principal (nuestra),Un partido del tour,"79,299",65,match_id


### Conclusión del paso 2
- **Fuente:** datos de Jeff Sackmann / Tennis Abstract, licencia CC BY-NC-SA 4.0, descargados del espejo en Hugging Face porque el repositorio original fue retirado.
- **Cuatro tablas unidas:** partidos del tour, qualys y challengers, jugadores y rankings semanales.
- **Una fila** = un partido del ATP Tour entre 2000 y junio de 2026. **Etiqueta** = `gana_A`.
- **El ranking se une hacia atrás**: solo el publicado en o antes de la fecha del torneo.
- **Las columnas de resultado y estadísticas del partido quedan marcadas**: no entran al modelo directamente, solo como historial.
 
| Tabla | Una fila es | Filas | Columnas | Llave |
|---|---|---|---|---|
| Partidos del tour | Un partido | 79.299 | 50 | tourney_id + match_num |
| Qualys y challengers | Un partido | 207.418 | 49 | tourney_id + match_num |
| Jugadores | Un jugador | 66.912 | 8 | player_id |
| Rankings | Un jugador en una semana | 3.114.367 | 4 | ranking_date + player |
| **Tabla principal (nuestra)** | **Un partido del tour** | **79.299** | **65** | **match_id** |
 
**Validación de las uniones**
- **Ninguna unión perdió ni duplicó filas:** la tabla principal tiene los mismos 79.299 partidos del paso 1.
- **Ranking:** el ranking unido coincide con el del archivo en el **95,1 %** de los partidos, y en la mitad de los casos se publicó el mismo día del torneo (mediana de 0 días). Esto confirma que el ranking del archivo es pre-partido, es decir, que no es una fuga. Además, la unión encontró ranking para **458 partidos** donde el archivo no lo traía.
- **Rankings viejos:** el máximo es de **7.935 días (unos 22 años)**, de un jugador que dejó de aparecer en el ranking y volvió a jugar mucho después. Un ranking tan viejo no describe al jugador de hoy, así que en la limpieza se trata como "sin ranking".
- **Edad:** la calculada desde la fecha de nacimiento difiere de la del archivo en **0,05 años** (mediana), unos 19 días, lo que valida la unión con jugadores.
- **Estatura:** falta en el **2,9 %** de los partidos, y la tabla de jugadores solo completa 7. Las dos fuentes comparten el mismo origen, así que el resto se imputa en el paso 3.
 
**Pendientes para la limpieza:** retiros y walkovers, superficie Carpet, rankings demasiado viejos, estatura faltante y el nivel "O"